* **Nguồn dataset:** https://drive.google.com/drive/folders/1xclbjHHK58zk2X6iqbvMPS2rcy9y9E0X

* **Mô tả:** Công bố khoa học: K. V. Nguyen, V. D. Nguyen, P. X. V. Nguyen, T. T. H. Truong and N. L. Nguyen, "UIT-VSFC: Vietnamese Students’ Feedback Corpus for Sentiment Analysis", KSE 2018.

Số lượng dữ liệu: khoảng hơn 16,000 câu và ngoài ra sử dụng thư viện sử dụng: gensim, chi tiết thư viện: https://pypi.org/project/gensim/



# 1 Import thư viện

In [7]:
import pandas as pd
from pyvi import ViTokenizer

# 2 Import dữ liệu

In [9]:
with open("UIT_VSFC/train/sents.txt", encoding="utf-8") as f:
    X_train = f.read().splitlines()

print(X_train[:5])

['slide giáo trình đầy đủ .', 'nhiệt tình giảng dạy , gần gũi với sinh viên .', 'đi học đầy đủ full điểm chuyên cần .', 'chưa áp dụng công nghệ thông tin và các thiết bị hỗ trợ cho việc giảng dạy .', 'thầy giảng bài hay , có nhiều bài tập ví dụ ngay trên lớp .']


# 3. Tokenize tiếng Việt

**Mô tả kĩ thuật:** Tiếng Việt là ngôn ngữ đa âm tiết, vì vậy một từ có thể gồm nhiều token. Nếu tách theo khoảng trắng, điều này không phản ánh đúng nghĩa của từ. Vì vậy cần sử dụng tokenizer cho tiếng Việt như PyVi để gộp các từ ghép:

In [10]:
from pyvi import ViTokenizer
tokenized_sentences = []
for s in X_train:
    tokens = ViTokenizer.tokenize(s).split()
    tokenized_sentences.append(tokens)
print(tokenized_sentences[:2])

[['slide', 'giáo_trình', 'đầy_đủ', '.'], ['nhiệt_tình', 'giảng_dạy', ',', 'gần_gũi', 'với', 'sinh_viên', '.']]


# 4. Train Word2Vec

**Mô tả:**

* Sau khi tokenization, bước tiếp theo là huấn luyện Word Embedding bằng Word2Vec.
* Word2Vec là mô hình học vector biểu diễn của từ dựa trên ngữ cảnh xuất hiện trong văn bản.
* Trong bài này, thư viện Gensim được sử dụng để huấn luyện Word2Vec.

In [11]:
from gensim.models import Word2Vec
import multiprocessing

**tạo model**

In [12]:
w2v_model = Word2Vec(
    vector_size=300,
    window=2,
    min_count=20,
    workers=multiprocessing.cpu_count()
)

Giải thích tham số:
| tham số | ý nghĩa |
|---------|---------|
| vector_size=300 | vector embedding 300 chiều |
| window=2 | context window |
| min_count=20 | bỏ từ hiếm |
| workers | số CPU |

# 5. Build vocabulary

**Mô tả:** Trước khi huấn luyện, mô hình cần xây dựng tập từ vựng (vocabulary) từ dữ liệu huấn luyện.

In [13]:
w2v_model.build_vocab(tokenized_sentences)
print("Vocabulary size:", len(w2v_model.wv))

Vocabulary size: 576


**Giải thích:** Điều này nghĩa là mô hình đã học được 576 từ khác nhau từ dataset.

# 6 Train model

**Mô tả:** Trong quá trình này, Word2Vec sẽ học cách biểu diễn mỗi từ dưới dạng một vector số sao cho các từ có ngữ nghĩa gần nhau sẽ có vector gần nhau.

In [14]:
w2v_model.train(
    tokenized_sentences,
    total_examples=w2v_model.corpus_count,
    epochs=30
)

(2134764, 3851460)

# 7. Kiểm tra vector của từ

**Mô tả:** Sau khi huấn luyện xong, ta có thể truy xuất vector của một từ bất kỳ.

In [15]:
print(w2v_model.wv["giảng_dạy"])

[-6.90959692e-02  9.86712947e-02  6.92039788e-01 -8.96533251e-01
  5.26114285e-01 -5.59749722e-01  4.82793301e-01  5.95911741e-01
  1.44371316e-01 -7.40179792e-02 -5.08042097e-01 -2.78078970e-02
 -5.62497020e-01  2.60709897e-02 -4.21695769e-01  2.36885503e-01
  1.14650741e-01  1.08611226e-01 -5.29929817e-01 -5.08225672e-02
 -2.02822611e-01  2.24985048e-01  3.21736187e-01  2.71510512e-01
 -4.12197083e-01  1.04071051e-01  7.75071979e-02 -4.80475485e-01
 -3.71243715e-01  1.57223299e-01  1.96497306e-01 -1.86722219e-01
 -1.58368319e-01 -1.76031217e-01  1.63712591e-01  2.53950357e-01
 -1.89708665e-01 -1.55424193e-01  2.15020135e-01 -1.02020897e-01
 -3.29354219e-02  3.81383359e-01 -5.27159750e-01 -1.86295584e-02
  4.37382907e-01  7.28883734e-03  3.62321824e-01  3.03520322e-01
 -2.76911587e-01 -8.44964087e-02 -1.21277504e-01 -3.29806894e-01
 -2.03664079e-01 -4.76804078e-02 -5.86988866e-01 -3.45715404e-01
 -5.12531459e-01 -1.38936341e-01  6.75816238e-01 -3.06104273e-01
 -3.34844947e-01 -6.47451

# 8. Tìm từ giống nhau (Similarity)

**Mô tả:** Một trong những ứng dụng quan trọng của Word Embedding là tìm các từ có ngữ nghĩa tương tự.

In [16]:
w2v_model.wv.most_similar("giảng_dạy")

[('dạy_học', 0.617289662361145),
 ('dạy', 0.5801670551300049),
 ('truyền_đạt', 0.47439923882484436),
 ('học_tập', 0.47091323137283325),
 ('giảng', 0.4519409239292145),
 ('giảng_giải', 0.42834338545799255),
 ('chu_đáo', 0.40622514486312866),
 ('thái_độ', 0.380109965801239),
 ('sôi_nổi', 0.37630578875541687),
 ('thay_đổi', 0.3706531226634979)]

**Giải thích:** Kết quả ở trên có nghĩa là bên trái là cột điểm tương đồng tương đương với mỗi từ nằm bên phải

# 9. Analogical reasoning

**Mô tả:** Word2Vec có thể thực hiện phép toán ngữ nghĩa trên vector từ.

In [17]:
w2v_model.wv.most_similar(
    positive=["giảng_dạy", "đồ_án"],
    negative=["kiểm_tra"],
    topn=1
)

[('dạy', 0.38322311639785767)]

**Giải thích:** Một đặc điểm thú vị của Word2Vec là khả năng thực hiện **phép toán ngữ nghĩa trên vector từ**.  
Điều này cho phép mô hình tìm ra các mối quan hệ giữa các từ thông qua phép cộng và trừ vector.

Trong thí nghiệm này, em thực hiện phép toán:

giảng_dạy + đồ_án − kiểm_tra

Ý nghĩa các tham số

* positive: danh sách các từ có vector được cộng vào phép toán.
* negative: danh sách các từ có vector được trừ khỏi phép toán.
* topn: số lượng kết quả tương tự nhất cần trả về.

Về mặt vector, phép toán này tương đương: **vector("giảng_dạy") + vector("đồ_án") - vector("kiểm_tra")**

# 10. Lưu Word Embedding

**Lưu dạng txt**

In [18]:
w2v_model.wv.save_word2vec_format(
    "w2v_vsfc.txt",
    binary=False
)

**Lưu dạng binary**

In [19]:
w2v_model.wv.save_word2vec_format(
    "w2v_vsfc.bin",
    binary=True
)

# 11. Kiểm tra embedding dimension

In [20]:
print(w2v_model.vector_size)

300


**Giải thích:** Kết quả 300 cho biết rằng mỗi từ trong mô hình Word2Vec được biểu diễn bằng một vector gồm 300 chiều.

# 12. Kiểm tra từ có trong vocabulary không

**Mô tả:** kiểm tra xem một từ có tồn tại trong vocabulary hay không.

In [21]:
print("giảng_dạy" in w2v_model.wv)

True


**Giải thích:**

* True: từ "giảng_dạy" tồn tại trong vocabulary của mô hình.
* False: từ đó không tồn tại trong vocabulary.

Nếu một từ không có trong vocabulary, mô hình sẽ không thể lấy vector embedding của từ đó.